In [ ]:
import pandas as pd 
from pandas import CategoricalDtype 
import numpy as np
import sqlite3
import plotnine as pm
import scipy.stats as stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


In [ ]:
intake_raw = pd.read_csv('/kaggle/input/datasets/jackdaoud/animal-shelter-analytics/Austin_Animal_Center_Intakes.csv')
outcome_raw = pd.read_csv('/kaggle/input/datasets/jackdaoud/animal-shelter-analytics/Austin_Animal_Center_Outcomes.csv')


def header():
    print('***' * 32)
    return


def imported_data_clean(df) -> pd.DataFrame: 
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_', regex = False) 
    df = df[~df.duplicated(subset=['animal_id', 'datetime'])].copy() 
    error_count = 0

    for column in df.columns:
        if df[column].dtype == 'object':
            try:
                df[column] = (
                    df[column].astype(str) .str.strip().str.lower())
            except AttributeError:
                print(f'Skipping non-string/mixed column: {column}')
                error_count += 1
                
    if error_count != 0:
        print(f'{error_count} columns not made into lowercase and stripped')
    return df 



def datetime_y_appointment_id(df) -> pd.DataFrame:
    if df['datetime'].astype(str).str.contains(r'AM|PM', case = False, regex = True).any():
        df['datetime'] = pd.to_datetime(df['datetime'], format = ('%m/%d/%Y %I:%M:%S %p'), errors = 'coerce') 
        
    else:
        df['datetime'] = pd.to_datetime(df['datetime'], format = '%Y-%m-%d %H:%M:%S', errors = 'coerce')

    df['std_date_time'] = df['datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
    df['appointment_id'] = df['animal_id'].astype(str) + '_' + df['std_date_time'].astype(str)
    
    #print(df['datetime'].dt.tz)
    #this came as none, so i can assume the times are local to austin
    #print(df['datetime'].dt.hour.value_counts().sort_index())
    return df



def clean_name(df) -> pd.DataFrame:
    df.columns = df.columns.str.replace(r'^name.*$', 'name', regex = True)
    
    df['name_given_at_intake'] = (df['name'].astype(str).str.startswith('*'))
    df['cln_name'] = df['name'].str.lstrip('*') 
    df['cln_name'] = df['cln_name'].astype(str)
    df['cln_name'] = df['cln_name'].replace(['nan', ''], np.nan)
    df['cln_name'] = df['cln_name'].fillna(df['animal_id'].astype(str))
    
    df = df.drop(columns = 'name', axis = 1)
    
    return df



def drop_intake_columns(df) -> pd.DataFrame:
    df = df.drop(columns = ['monthyear', 'found_location','breed', 'color'])
    return df



def shift_extraction(df, df_name) -> pd.DataFrame:
    shift_conditions = [
        (df['datetime'].dt.hour >= 7) & (df['datetime'].dt.hour < 16),
        (df['datetime'].dt.hour >= 16) & (df['datetime'].dt.hour <= 23),
        (df['datetime'].dt.hour >= 0) & (df['datetime'].dt.hour < 7)
    ]
    shift_choices = ['day', 'swing', 'overnight']
    df['shift'] = np.select(shift_conditions, shift_choices, default='unknown')
    return df



def intake_type_clean(df) -> pd.DataFrame:
    header()
    #print(f'unique values: {df['intake_type'].unique()}')
    #print(f'\n\n\nnull values: {df['intake_type'].isna().sum()}')

    #this information looks clean and that makes sense as this is recorded by the staff at time of intake
    return df



def intake_condition_clean(df) -> pd.DataFrame:
    medical_conditions = ['sick', 'injured', 'medical', 'aged']
    behavior_conditions = ['feral', 'behavior']
    reproductive_conditions = ['pregnant', 'nursing']
    routine_conditions = ['normal']

    df['pregnant_o_nursing'] = np.where(df['intake_condition'].isin(reproductive_conditions), True, False)

    conditions = [
        df['intake_condition'].isin(medical_conditions) | df['intake_condition'].isin(reproductive_conditions),
        df['intake_condition'].isin(behavior_conditions),
        df['intake_condition'].isin(routine_conditions)
    ]

    choices = [
        'medical' ,
        'behavior',
        'routine'
    ]

    df['intake_reason'] = np.select(conditions, choices, default = 'unknown')
    df = df.drop(columns = ['intake_condition'], axis = 1)
    return df




def perros_y_gatos(df) -> pd.DataFrame:
    df = df.rename(columns = {
        'animal_type' : 'spp'
    })
    df = df[df['spp'].isin(['dog', 'cat'])]
    df['spp'] = df['spp'].replace({'cat': 'fel', 'dog': 'k9'})
    return df


    
def age_clean(df) -> pd.DataFrame:
    df.columns = df.columns.str.replace(r'^age.*$', 'cln_age', regex = True)
    df['age_num'] = df['cln_age'].str.split(' ', n = 1, expand = True)[0]
    df['age_num'] = df['age_num'].replace(['nan', ''], np.nan)
    df['cln_age'] = df['cln_age'].replace(['nan', ''], np.nan)
    df['age_num'] = pd.to_numeric(df['age_num'])
    df['age_num'] = df['age_num'].where(df['age_num'] > 0, np.nan)
        
    conditions = [
        (df['cln_age'].str.contains(r'months?', case = False, na = False, regex = True)), 
        (df['cln_age'].str.contains(r'years?', case = False, na = False, regex = True)),
        (df['cln_age'].str.contains(r'weeks?', case = False, na = False, regex = True)),
        (df['cln_age'].str.contains(r'days?', case = False, na = False, regex = True))
    ]

    choices = [
        (df['age_num'] / 12), #converts the age in months to years
        (df['age_num']), #keeps the same age as it is in years
        (df['age_num'] / 52.1786), #converts from weeks to years
        (df['age_num'] / 365.25) #converts frem days to years
    ]

    df['age'] = np.select(conditions, choices, default = np.nan)
    df = df.drop(columns = ['age_num', 'cln_age'], axis = 1)

    return df




def lifecycle(df, df_name) -> pd.DataFrame:
    conditions = [
        (df['age'] < 1), # less than 1 year
        (df['age'] >= 1) & (df['age'] < 3), # 1 to less than 3 years
        (df['age'] >= 3) & (df['age'] < 7), # 3 to less than 7 years
        (df['age'] >= 7) # 7 years and older
    ]

    choices = [
        'juvenile',
        'young adult',
        'adult',
        'senior'
    ]

    df['lifecycle_stage'] = np.select(conditions, choices, default = 'unknown')
    df = df.drop(columns = ['age'], axis = 1)
    if df_name == 'intake':
        df = df.rename(columns = {'lifecycle_stage': 'age_in'})
    else:
        df = df.rename(columns = {'lifecycle_stage': 'age_out'})
    return df



def clean_sex(df, df_name) -> pd.DataFrame:
    df.columns = df.columns.str.replace(r'^sex.*$', 'sex', regex = True)

    condition = [
        df['sex'] == 'intact male',
        df['sex'] == 'intact female',
        df['sex'] == 'neutered male',
        df['sex'] == 'spayed female',
    ]

    choice = [
        'm',
        'f',
        'n',
        's'
    ]

    df['sex'] = np.select(condition, choice, default = 'unknown')
    
    if df_name == 'intake':
        df = df.rename(columns = {'sex': 'sex_in'})
    else:
        df = df.rename(columns = {'sex': 'sex_out'})
    return df



def table_check(df, df_name) -> pd.DataFrame:
    '''
    Checks and prints the count of null (NaN) values for every column in a Pandas DataFrame.
    '''
    header()
    print(f'Beginning errorcheck for: {df_name} ***')
    print(f'\n--- Null Value Check ---')
    for column in df.columns: 
        null_count = df[column].isna().sum() 
        print(f'{column} nulls: {null_count}')
    print('\n')
    
    print(f'--- Integrity Check ---')
    if 'appointment_id' in df.columns:
        duplicate_count = df['appointment_id'].duplicated().sum()
        print(f'Total appointment_id duplicates: {duplicate_count}')
    else:
        print("WARNING: 'appointment_id' not found for duplication check.")
    print('\n')
    print(f'\n--- Data Type Audit (DF.dtypes) ---')
    print(df.dtypes)
    
    print(f'\n--- Numerical Sanity Check (DF.describe) ---')
    
    numeric_cols = df.select_dtypes(include = ['int64', 'float64', 'Int64', 'int32']).columns
    if not numeric_cols.empty:
        print(df[numeric_cols].describe())
    else:
        print('No standard numerical columns found for description.')

    print(f'\n--- Categorical Value Audit (Top 10 Counts) ---')
    object_cols = df.select_dtypes(include=['object']).columns

    for column in object_cols:
        print(f'\n{column.upper()}:')
        print(df[column].value_counts().nlargest(10))
    
    print('\n\n\n')
    
    return df


def drop_out_columns(df) -> pd.DataFrame:
    df = df.drop(columns = ['monthyear', 'date_of_birth','breed', 'color'])
    return df



def clean_outcome_type(df) -> pd.DataFrame:
    unknown_outcome = ['nan', 'missing']
    alive_outcome = ['rto-adopt', 'adoption', 'return to owner']
    administrative_outcome = ['transfer', 'relocate']
    deceased_outcome = ['euthanasia', 'died', 'disposal']
    
    conditions = [
        df['outcome_type'].isin(unknown_outcome),
        df['outcome_type'].isin(alive_outcome),
        df['outcome_type'].isin(administrative_outcome),
        df['outcome_type'].isin(deceased_outcome)
    ]

    choices = [
        'unknown',
        'alive',
        'admin',
        'deceased'
    ]

    df['outcome_category'] = np.select(conditions, choices, default = 'unknown')
    df = df.drop(columns = 'outcome_type', axis = 1)
    return df



def clean_outcome_subtype(df) -> pd.DataFrame:
    location_subtype = ['in kennel', 'offsite', 'at vet', 'barn', 'enroute', 'in surgery'] #the animal's specific physical location or temporary status
    behavior_subtype = ['suffering', 'medical', 'aggressive', 'rabies risk', 'behavior'] #indicates the reason for a specific outcome (often euthanasia or specialized treatment)
    program_subtype = ['partner', 'underage', 'foster', 'in foster', 'snr', 'scrp', 'prc'] #subtypes related to specific shelter programs or transfer partners
    admin_subtype = ['field', 'possible theft', 'customer s', 'court/investigation', 'emer'] #Miscellaneous or administrative details
    unknown_subtype = ['nan']

    conditions = [
        df['outcome_subtype'].isin(location_subtype),
        df['outcome_subtype'].isin(behavior_subtype),
        df['outcome_subtype'].isin(program_subtype),
        df['outcome_subtype'].isin(admin_subtype),
        df['outcome_subtype'].isin(unknown_subtype)
    ]

    choices = [
        'location' ,
        'behavior',
        'program',
        'admin',
        'unknown'
    ]

    df['outcome_subcategory'] = np.select(conditions, choices, default = 'unknown')
    df = df.drop(columns = 'outcome_subtype', axis = 1)
    return df








def create_intake_table(df, df_name) -> pd.DataFrame:
    header()
    print(f'Beginning to create intake table')
    return ( 
        df
        .pipe(imported_data_clean)
        .pipe(datetime_y_appointment_id)
        .pipe(clean_name)
        .pipe(drop_intake_columns)
        .pipe(shift_extraction, df_name)
        .pipe(intake_type_clean)
        .pipe(intake_condition_clean)
        .pipe(perros_y_gatos)
        .pipe(age_clean)
        .pipe(lifecycle, df_name)
        .pipe(clean_sex, df_name)
        .pipe(table_check, df_name)
    )

print('\n\n')
def create_outcome_table(df, df_name) -> pd.DataFrame:
    print('\n\n\n')
    header()
    print(f'Beginning to create outcome table')
    
    return (
        df
        .pipe(imported_data_clean)
        .pipe(datetime_y_appointment_id)
        .pipe(shift_extraction, df_name)
        .pipe(drop_out_columns)
        .pipe(perros_y_gatos)
        .pipe(clean_name)
        .pipe(clean_sex, df_name)
        .pipe(age_clean)
        .pipe(lifecycle, df_name)
        .pipe(clean_outcome_type)
        .pipe(clean_outcome_subtype)
        .pipe(table_check, df_name)
    )






In [ ]:
intake_cleaned = create_intake_table(intake_raw, 'intake')
intake_cleaned.to_csv('intake_sorted.csv', index = False)

In [ ]:
outcome_cleaned = create_outcome_table(outcome_raw, 'outcome')
outcome_cleaned.to_csv('outcome_sorted.csv', index = False)

In [ ]:
weather_data = pd.read_csv('/kaggle/input/datasets/hannashuraieva/austin-weather-data-2000-2023/austin_weather_data_sorted.csv')

def make_datetime(df) -> pd.DataFrame:
    df['datetime'] = pd.to_datetime(df['datetime'])
    return df

def drop_dupes(df) -> pd.DataFrame:
    df = df[~df.duplicated(subset=['datetime'])].copy()
    return df

def collimate_data(df) -> pd.DataFrame:
    df = df[df['datetime'].dt.year.between(2013, 2021)].copy()
    return df

def units(df) -> pd.DataFrame:
    '''
    according to the metadata, this was collected at https://www.visualcrossing.com/  These units i am
    adding came from the output noted when that site was visited 5/3/2026 at 11:30a
    '''
    df = df.rename(columns = {
        'tempmax' : 'temp_max(\u00b0F)', 
        'tempmin' : 'temp_min(\u00b0F)', 
        'temp' : 'temp(\u00b0F)', 
        'feelslikemax' : 'feels_like_max(\u00b0F)',
        'feelslikemin' : 'feels_like_min(\u00b0F)', 
        'feelslike' : 'feels_like(\u00b0F)', 
        'dew' : 'dew_point(\u00b0F)', 
        'humidity' : 'humidity(%)', 
        'precip' : 'precip(in)', 
        'precipprob' : 'precip_prop(%)',
        'precipcover' : 'precip_cover(%)', 
        'preciptype' : 'precip_type',
        'snow' : 'snow(in)', 
        'snowdepth' : 'snow_depth(in)', 
        'windgust' : 'wind_gust(mph)',
        'windspeed' : 'wind_speed(mph)', 
        'winddir' : 'wind_dir(\u00b0)', 
        'sealevelpressure' : 'sea_level_pressure(mb)', 
        'cloudcover' : 'cloud_cover', 
        'visibility' : 'visibility(mi)',
        'solarradiation' : 'solar_radiation(W/m\u00b2)', 
        'solarenergy' : 'solar_energy(MJ/m\u00b2)', 
        'uvindex' : 'uv_index', 
        'severerisk' : 'severe_risk'
    })
    return df



def classify_weather_data(df) -> pd.DataFrame:
    #first im going to look at the data types and the columns
    print(df.columns)
    print(df.dtypes)
    #looks good for now...
    return df

def moonphase_cat(df) -> pd.DataFrame:
    conditions = [
        df['moonphase'] == 0,
        (df['moonphase'] > 0) & (df['moonphase'] < 0.25),
        df['moonphase'] == 0.25,
        (df['moonphase'] > 0.25) & (df['moonphase'] < 0.5),
        df['moonphase'] == 0.5,
        (df['moonphase'] > 0.5) & (df['moonphase'] < 0.75),
        df['moonphase'] == 0.75,
        (df['moonphase'] > 0.75) & (df['moonphase'] < 1.0)
        
    ]


    choices = [
        'new_moon', #0
        'waxing_crescent', #0-24 
        'first_quarter', #25 
        'waxing_gibbous', #26-49 
        'full', #50
        'waning_gibbous', #51-74
        'third_quarter', #75
        'waning_crescent' #76-99
    ]

    df['cln_moonphase'] = np.select(conditions, choices, default = 'unknown')
    return df

def wind_direction(df) -> pd.DataFrame:
    conditions = [
        (df['wind_dir(\u00b0)'] > 348.75) | (df['wind_dir(\u00b0)'] <= 11.25),
        df['wind_dir(\u00b0)'].between(11.25, 33.75, inclusive='right'),
        df['wind_dir(\u00b0)'].between(33.75, 56.25, inclusive='right'),
        df['wind_dir(\u00b0)'].between(56.25, 78.75, inclusive='right'),
        df['wind_dir(\u00b0)'].between(78.75, 101.25, inclusive='right'),
        df['wind_dir(\u00b0)'].between(101.25, 123.75, inclusive='right'),
        df['wind_dir(\u00b0)'].between(123.75, 146.25, inclusive='right'),
        df['wind_dir(\u00b0)'].between(146.25, 168.75, inclusive='right'),
        df['wind_dir(\u00b0)'].between(168.75, 191.25, inclusive='right'),
        df['wind_dir(\u00b0)'].between(191.25, 213.75, inclusive='right'),
        df['wind_dir(\u00b0)'].between(213.75, 236.25, inclusive='right'),
        df['wind_dir(\u00b0)'].between(236.25, 258.75, inclusive='right'),
        df['wind_dir(\u00b0)'].between(258.75, 281.25, inclusive='right'),
        df['wind_dir(\u00b0)'].between(281.25, 303.75, inclusive='right'),
        df['wind_dir(\u00b0)'].between(303.75, 326.25, inclusive='right'),
        df['wind_dir(\u00b0)'].between(326.25, 348.75, inclusive='right')
    ]

    choices = [
        'n',
        'nne',
        'ne',
        'ene',
        'e',
        'ese',
        'se',
        'sse',
        's',
        'ssw',
        'sw',
        'wsw',
        'w',
        'wnw',
        'nw',
        'nnw'
    ]
    df['wind_direction'] = np.select(conditions, choices, default = 'unknown')
    return df
    
def create_weather_table(df, df_name) -> pd.DataFrame:
    header()
    print(f'Beginning to create weather table')
    return ( 
        df
        .pipe(make_datetime)
        .pipe(drop_dupes)
        .pipe(collimate_data)
        .pipe(units)
        .pipe(classify_weather_data)
        .pipe(moonphase_cat)
        .pipe(wind_direction)
        .pipe(table_check, df_name)
        
    )


In [ ]:
weather_sorted = create_weather_table(weather_data, 'weather')
weather_sorted.to_csv('weather_sorted.csv', index = False)